# Módulo 9. Comparación con una red neuronal

Con el propósito de comparar diferentes enfoques de aprendizaje automático, se entrena una red neuronal multicapa (MLPRegressor) utilizando el mismo conjunto de entrenamiento empleado para el modelo Random Forest.

Posteriormente se comparan las métricas obtenidas para determinar cuál algoritmo presenta un mejor desempeño en la predicción de la deserción escolar.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "dataset_modelado.csv"
)

print(df.shape)

df.head()

(15707, 85)


,AÑO,CÓDIGO_MUNICIPIO,MUNICIPIO,CÓDIGO_DEPARTAMENTO,DEPARTAMENTO,CÓDIGO_ETC,ETC,POBLACIÓN_5_16,TASA_MATRICULACIÓN_5_16,COBERTURA_NETA,...,REPITENCIA_MEDIA_ETC,BENEFICIARIOS_PAE,BRECHA_COBERTURA,BRECHA_APROBACION,INDICE_EFICIENCIA,PRESION_SISTEMA,DIGITALIZADO,TAM_GRUPO_NORMALIZADO,PESO_MUNICIPIO_ETC,PANDEMIA
0,2024,5004,Abriaquí,5,Antioquia,3758.0,Antioquia (ETC),499.0,56.11,56.11,...,3.35,NaN,5.81,99.36,11.418099,8.893245,0,NaN,0.001007,0
1,2024,15204,Cómbita,15,Boyacá,3769.0,Boyacá (ETC),1862.0,95.33,95.33,...,3.21,NaN,96.18,91.52,6.821739,19.532151,0,NaN,0.011952,0
2,2024,99773,Cumaribo,99,Vichada,3832.0,Vichada (ETC),25239.0,50.70,50.70,...,4.08,NaN,7.04,64.06,1.940284,497.810651,0,NaN,0.750245,0
3,2024,99624,Santa Rosalía,99,Vichada,3832.0,Vichada (ETC),1157.0,81.42,81.42,...,4.08,NaN,9.16,78.44,3.176015,14.210268,0,NaN,0.034393,0
4,2024,99524,La Primavera,99,Vichada,3832.0,Vichada (ETC),2645.0,90.96,90.96,...,4.08,NaN,8.17,77.18,3.169123,29.078716,0,NaN,0.078624,0


In [3]:
from sklearn.model_selection import train_test_split

# Variable objetivo
y = df["DESERCIÓN"]

columnas_excluir = [
    "DESERCIÓN",
    "MUNICIPIO",
    "DEPARTAMENTO",
    "ETC",
    "CÓDIGO_MUNICIPIO",
    "CÓDIGO_DEPARTAMENTO",

    "DESERCIÓN_TRANSICIÓN",
    "DESERCIÓN_PRIMARIA",
    "DESERCIÓN_SECUNDARIA",
    "DESERCIÓN_MEDIA",

    "DESERCIÓN_ETC",
    "DESERCIÓN_TRANSICIÓN_ETC",
    "DESERCIÓN_PRIMARIA_ETC",
    "DESERCIÓN_SECUNDARIA_ETC",
    "DESERCIÓN_MEDIA_ETC"
]

X = df.drop(columns=columnas_excluir)

if "ETC_ETC" in X.columns:
    X = X.drop(columns=["ETC_ETC"])

X = X.fillna(X.median(numeric_only=True))

mask = y.notna()

X = X.loc[mask]
y = y.loc[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(12452, 69)
(3113, 69)


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        max_iter=500,
        random_state=42
    ))
])

mlp.fit(X_train, y_train)

print("✅ Red neuronal entrenada correctamente.")

✅ Red neuronal entrenada correctamente.


In [5]:
y_pred = mlp.predict(X_test)

print(f"MAE : {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")
print(f"R²  : {r2_score(y_test, y_pred):.3f}")

MAE : 0.117
RMSE: 0.394
R²  : 0.967


## 9.1 Comparación de modelos

Se comparó el desempeño del modelo Random Forest con una red neuronal multicapa (MLPRegressor), utilizando el mismo conjunto de entrenamiento y prueba.

Las métricas obtenidas fueron las siguientes:

In [6]:
comparacion_modelos = pd.DataFrame({
    "Modelo": ["Random Forest", "Red neuronal (MLP)"],
    "MAE": [0.126, 0.117],
    "RMSE": [0.372, 0.394],
    "R²": [0.970, 0.967]
})

comparacion_modelos

,Modelo,MAE,RMSE,R²
0,Random Forest,0.126,0.372,0.970
1,Red neuronal (MLP),0.117,0.394,0.967


Aunque la red neuronal obtuvo un error absoluto medio ligeramente inferior, el modelo Random Forest presentó un menor error cuadrático medio y un mayor coeficiente de determinación (R²), además de ofrecer una mejor interpretabilidad mediante el análisis de importancia de variables.

Por estas razones, el modelo Random Forest fue seleccionado como el modelo final para el sistema de predicción de la deserción escolar.